## Kelompok 2 (Kelas 24-01)

Anggota Kelompok (Alfabetis):
1. Azis Al Risal
2. Bagas Dwi Saputra
3. Desta Ega Fatima
4. Jonathan Steve Roland
5. Nayla Khairusyi Shabrina

---

> _Dev Note_
>
> _Sebelum menjalankan notebook ini, pastikan semua pustaka yang dibutuhkan sudah terpasang menggunakan command: `pip install -r requirements.txt`_

##### 1. Persiapan Awal dan Impor Data
Tahap persiapan memuat seluruh pustaka Python yang dibutuhkan untuk analisis data dan machine learning:
- `pandas` untuk memuat, membaca, dan memanipulasi data tabular.
- `matplotlib.pyplot` dan `seaborn` untuk visualisasi data dan pembuatan grafik.
- `StandardScaler` dari `sklearn.preprocessing` untuk menyetarakan skala fitur kontinu.
- `KMeans` dari `sklearn.cluster` untuk algoritma pengelompokan (clustering) tanpa supervisi.

Dataset dibaca dari file `tumor-data.csv` dan ditampilkan informasi strukturnya.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Membaca dataset tumor
df = pd.read_csv("tumor-data.csv")

print("[INFO] 5 BARIS PERTAMA DATASET")
display(df.head())
print("\n[INFO] INFORMASI STRUKTUR DATA")
df.info()
print("\n[INFO] STATISTIK DESKRIPTIF DATASET")
display(df.describe())

##### 2. Exploratory Data Analysis (EDA) Sebelum Preprocessing
Tahap EDA awal dilakukan untuk mengenali karakteristik data mentah sebelum dilakukan proses clustering:
1. **Histogram Perbedaan Skala**: Memperlihatkan disparitas rentang nilai yang sangat drastis antar fitur (misalnya `mean area` bernilai ratusan hingga ribuan, sedangkan `mean smoothness` bernilai desimal sangat kecil).
2. **Heatmap Korelasi**: Memetakan hubungan korelasi linier antar 10 fitur morfologi pertama.
3. **Boxplot Outlier**: Mendeteksi keberadaan nilai pencilan (*outliers*) sebelum data distandarisasi.
4. **Scatter Plot**: Meninjau persebaran hubungan antara dua fitur utama (`mean radius` vs `mean texture`).

In [ ]:
print("[INFO] MEMBUAT GRAFIK PRE-EDA...")

# 1. Histogram Perbedaan Skala
plt.figure(figsize=(8, 5))
sns.histplot(df["mean area"], color="red", alpha=0.5, label="Mean Area")
sns.histplot(df["mean smoothness"], color="blue", alpha=0.5, label="Mean Smoothness")
sns.histplot(df["mean radius"], color="green", alpha=0.5, label="Mean Radius")
plt.legend()
plt.title("Perbedaan Skala Angka Pada Data Tumor")
plt.tight_layout()
plt.show()

# 2. Heatmap Korelasi 10 Fitur Pertama
plt.figure(figsize=(10, 8))
sns.heatmap(df.iloc[:, :10].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Heatmap Korelasi 10 Fitur Pertama Data Tumor")
plt.tight_layout()
plt.show()

# 3. Boxplot Deteksi Outlier
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[["mean radius", "mean texture", "mean perimeter", "mean area"]])
plt.title("Boxplot Deteksi Outlier (Sebelum Preprocessing)")
plt.tight_layout()
plt.show()

# 4. Scatter Plot: Mean Radius vs Mean Texture
plt.figure(figsize=(8, 5))
sns.scatterplot(x=df["mean radius"], y=df["mean texture"], color="purple", alpha=0.7)
plt.title("Persebaran Data: Mean Radius vs Mean Texture")
plt.tight_layout()
plt.show()

##### 3. Preprocessing Data (Feature Scaling)
Algoritma K-Means berbasis perhitungan jarak Euclidean. Jika data memiliki skala nilai yang berbeda jauh, fitur dengan angka besar akan mendominasi perhitungan jarak dan menenggelamkan fitur lain yang bernilai kecil.

Oleh karena itu, seluruh fitur dinormalisasi menggunakan `StandardScaler` sehingga memiliki nilai rata-rata (*mean*) = 0 dan variansi (*variance*) = 1.

In [ ]:
# Standarisasi data menggunakan StandardScaler
scaler = StandardScaler()
array_scaled = scaler.fit_transform(df)
df_scaled = pd.DataFrame(array_scaled, columns=df.columns)

print("[INFO] STANDARISASI SELESAI. VISUALISASI PASCA-SCALING:")

# Histogram Pasca-Standarisasi
plt.figure(figsize=(8, 5))
sns.histplot(df_scaled["mean area"], color="red", alpha=0.5, label="Mean Area")
sns.histplot(df_scaled["mean smoothness"], color="blue", alpha=0.5, label="Mean Smoothness")
sns.histplot(df_scaled["mean radius"], color="green", alpha=0.5, label="Mean Radius")
plt.legend()
plt.title("Distribusi Skala Data Tumor (Sesudah StandardScaler)")
plt.tight_layout()
plt.show()

# Boxplot Pasca-Standarisasi
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_scaled[["mean radius", "mean texture", "mean perimeter", "mean area"]])
plt.title("Boxplot Deteksi Outlier (Sesudah StandardScaler)")
plt.tight_layout()
plt.show()

##### 4. Hyperparameter Tuning (Metode Elbow)
Untuk menentukan jumlah kluster ($K$) yang paling optimal secara matematis, kita menggunakan **Metode Elbow**.

Proses ini menghitung nilai *inertia* (Within-Cluster Sum of Squares / WCSS) untuk jumlah kluster $K=1$ hingga $K=10$. Titik siku (*elbow point*) di mana penurunan inersia mulai melambat menunjukkan jumlah kluster terbaik.

In [ ]:
# Menghitung inersia untuk K = 1 sampai 10
inertia = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(array_scaled)
    inertia.append(kmeans.inertia_)

# Plot Kurva Metode Elbow
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker="o", linestyle="--", color="green")
plt.title("Metode Elbow untuk Menentukan K Optimal")
plt.xlabel("Jumlah Kluster (K)")
plt.ylabel("Inersia (Error Level)")
plt.xticks(range(1, 11))
plt.grid(True)
plt.tight_layout()
plt.show()

##### 5. Final Clustering (K=2) dan Profiling Karakteristik
Berdasarkan analisis metode elbow, titik pembelokan optimal berada pada **$K=2$**, yang selaras dengan dua klasifikasi klinis umum pada tumor (jinak / *benign* vs ganas / *malignant*).

Model K-Means akhir dilatih dengan parameter `n_clusters=2`. Setiap baris data diberikan label kluster, disimpan ke berkas `clustered.csv`, dan dianalisis nilai rata-rata fiturnya (*profiling*).

In [ ]:
# Eksekusi K-Means Final dengan K=2
kmeans_final = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(array_scaled)

# Menyematkan label kluster ke dataframe
df_clustered = df.copy()
df_clustered["cluster"] = clusters

# Menyimpan hasil ke clustered.csv
df_clustered.to_csv("clustered.csv", index=False)
print("[INFO] Hasil klusterisasi berhasil disimpan ke 'clustered.csv'\n")

print("[INFO] Total Anggota Masing-Masing Kluster:")
print(df_clustered["cluster"].value_counts())
print("\n" + "="*50)

print("[INFO] Profil Karakteristik Rata-Rata Fitur Per Kluster:")
cluster_profile = df_clustered.groupby("cluster").mean()
display(cluster_profile[["mean radius", "mean texture", "mean perimeter", "mean area", "mean concavity"]])

##### 6. Kesimpulan Akhir
1. **Jumlah Kluster Optimal**: Evaluasi metode elbow membuktikan bahwa pemisahan terbaik tercapai pada $K=2$.
2. **Karakteristik Kluster**:
   - Salah satu kluster memiliki rata-rata dimensi sel yang jauh lebih besar (`mean radius`, `mean perimeter`, `mean area`, dan `mean concavity` lebih tinggi), merefleksikan pola tumor ganas (*malignant*).
   - Kluster lainnya memiliki ukuran sel yang lebih kecil dan homogen, merefleksikan pola tumor jinak (*benign*).
3. **Efektivitas Scaling**: Penggunaan `StandardScaler` sangat krusial dalam menyetarakan pengaruh seluruh 30 fitur morfologi pada penentuan kluster K-Means.